# ML Challenge - Fault Detection
binary classification on sensor data (47 features) to detect faulty vs normal devices

In [8]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score, f1_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

## load data

In [9]:
train = pd.read_csv('TRAIN.csv')
test = pd.read_csv('TEST.csv')

print(train.shape, test.shape)
print('missing train:', train.isnull().sum().sum())
print('missing test:', test.isnull().sum().sum())
train['Class'].value_counts()

(43776, 48) (10944, 48)
missing train: 0
missing test: 0


Class
0    26465
1    17311
Name: count, dtype: int64

## preprocessing

In [18]:
# drop dupes
before = len(train)
train = train.drop_duplicates()
print(f'dropped {before - len(train)} duplicates, now {len(train)}')

# handle ID column - TEST.csv may or may not have it
if 'ID' in test.columns:
    test_ids = test['ID']
else:
    test_ids = pd.RangeIndex(1, len(test) + 1)
    print('no ID column found, using 1..N')

feature_cols = [c for c in train.columns if c != 'Class']
test_feature_cols = [c for c in test.columns if c not in ('ID', 'Class')]
print(f'features: {len(feature_cols)}, test features: {len(test_feature_cols)}')

dropped 0 duplicates, now 43038
no ID column found, using 1..N
features: 47, test features: 47


## feature engineering

In [19]:
def make_features(df, feat_cols):
    data = df[feat_cols].copy()
    
    # row-wise stats (using numpy for speed)
    vals = data[feat_cols].values
    data['feat_mean'] = vals.mean(axis=1)
    data['feat_std'] = vals.std(axis=1)
    data['feat_min'] = vals.min(axis=1)
    data['feat_max'] = vals.max(axis=1)
    data['feat_range'] = data['feat_max'] - data['feat_min']
    
    # log transform heavy-tailed features
    for col in ['F30','F31','F32','F33','F34','F35','F36','F37','F38']:
        data[f'{col}_log'] = np.log1p(data[col])
    
    # interaction terms between top correlated features
    data['F01_x_F09'] = data['F01'] * data['F09']
    data['F01_x_F29'] = data['F01'] * data['F29']
    data['F19_div_F21'] = data['F19'] / (data['F21'] + 1e-8)
    data['F05_div_F06'] = data['F05'] / (data['F06'] + 1e-8)
    data['F25_x_F27'] = data['F25'] * data['F27']
    data['F07_x_F08'] = data['F07'] * data['F08']
    
    # group sums
    data['sum_F01_F09'] = data[['F01','F02','F03','F04','F05','F06','F07','F08','F09']].sum(axis=1)
    data['sum_F10_F18'] = data[['F10','F11','F12','F13','F14','F15','F16','F17','F18']].sum(axis=1)
    data['sum_F19_F29'] = data[['F19','F20','F21','F22','F23','F24','F25','F26','F27','F28','F29']].sum(axis=1)
    data['sum_F30_F38'] = data[['F30','F31','F32','F33','F34','F35','F36','F37','F38']].sum(axis=1)
    data['sum_F39_F47'] = data[['F39','F40','F41','F42','F43','F44','F45','F46','F47']].sum(axis=1)
    
    return data

In [20]:
X_eng = make_features(train, feature_cols)
X_test_eng = make_features(test, test_feature_cols)

X = X_eng.values
y = train['Class'].values
X_test = X_test_eng.values

print(f'features: {X.shape[1]}, train: {X.shape[0]}, test: {X_test.shape[0]}')

features: 72, train: 43038, test: 10944


## models
reduced n_estimators + higher learning rate to keep things fast on colab. lgbm is super fast anyway, xgb hist mode also quick enough

In [21]:
scale_pos = y[y==0].shape[0] / y[y==1].shape[0]

# reduced depth + stronger regularization to avoid overfitting
xgb1 = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.7,
    colsample_bytree=0.5,
    reg_alpha=1.0,
    reg_lambda=5.0,
    scale_pos_weight=scale_pos,
    min_child_weight=10,
    gamma=0.3,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss',
    tree_method='hist'
)

lgbm1 = LGBMClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.7,
    colsample_bytree=0.5,
    reg_alpha=1.0,
    reg_lambda=5.0,
    is_unbalance=True,
    min_child_samples=50,
    num_leaves=31,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

lgbm2 = LGBMClassifier(
    n_estimators=250,
    max_depth=6,
    learning_rate=0.08,
    subsample=0.7,
    colsample_bytree=0.6,
    reg_alpha=0.5,
    reg_lambda=3.0,
    is_unbalance=True,
    min_child_samples=40,
    num_leaves=31,
    random_state=123,
    n_jobs=-1,
    verbose=-1
)

print('models ready (regularized to reduce overfitting)')

models ready (regularized to reduce overfitting)


## quick CV check
3-fold instead of 5, just to get a rough idea. no need to waste 40 min on CV

In [22]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

for name, mdl in [('XGBoost', xgb1), ('LightGBM-1', lgbm1), ('LightGBM-2', lgbm2)]:
    scores = cross_val_score(mdl, X, y, cv=cv, scoring='accuracy', n_jobs=-1)
    print(f'{name:12s}  acc={scores.mean():.5f} (+/-{scores.std():.5f})')

XGBoost       acc=0.96601 (+/-0.00043)
LightGBM-1    acc=0.96536 (+/-0.00063)
LightGBM-2    acc=0.97823 (+/-0.00065)


## train ensemble on full data
soft voting with the 3 models — no stacking CV needed, this is fast and works well

In [23]:
voting = VotingClassifier(
    estimators=[
        ('xgb', xgb1),
        ('lgbm1', lgbm1),
        ('lgbm2', lgbm2)
    ],
    voting='soft',
    n_jobs=-1
)

print('training ensemble...')
voting.fit(X, y)
print('done')

training ensemble...
done


## predict and save

In [24]:
preds = voting.predict(X_test)

print(f'total: {len(preds)}')
print(f'class 0 (normal): {(preds==0).sum()}')
print(f'class 1 (faulty): {(preds==1).sum()}')

total: 10944
class 0 (normal): 6688
class 1 (faulty): 4256


In [25]:
# save FINAL.csv (ID + CLASS)
output = pd.DataFrame({'ID': test_ids, 'CLASS': preds.astype(int)})
output.to_csv('FINAL.csv', index=False)

# save TEST.csv with predicted Class column added
test_out = test[test_feature_cols].copy()
test_out['Class'] = preds.astype(int)
test_out.to_csv('TEST.csv', index=False)

print(f'saved FINAL.csv ({len(output)} rows) and TEST.csv with Class column')
output.head(10)

saved FINAL.csv (10944 rows) and TEST.csv with Class column


,ID,CLASS
0,1,1
1,2,0
2,3,1
3,4,0
4,5,0
5,6,1
6,7,0
7,8,1
8,9,1
9,10,0


## model evaluation
holdout split (80/20) to check all metrics

In [26]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, classification_report,
                             confusion_matrix, roc_auc_score, matthews_corrcoef)

# 80/20 stratified split
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# train ensemble on 80%
eval_ens = VotingClassifier(
    estimators=[('xgb', xgb1), ('lgbm1', lgbm1), ('lgbm2', lgbm2)],
    voting='soft', n_jobs=-1
)
eval_ens.fit(X_tr, y_tr)
y_pred = eval_ens.predict(X_val)
y_prob = eval_ens.predict_proba(X_val)[:, 1]

# all metrics
print('=== Model Evaluation (20% Holdout) ===')
print(f'Accuracy       : {accuracy_score(y_val, y_pred):.5f}')
print(f'F1 Score       : {f1_score(y_val, y_pred):.5f}')
print(f'Precision      : {precision_score(y_val, y_pred):.5f}')
print(f'Recall         : {recall_score(y_val, y_pred):.5f}')
print(f'ROC-AUC        : {roc_auc_score(y_val, y_prob):.5f}')
print(f'MCC            : {matthews_corrcoef(y_val, y_pred):.5f}')
print()

# confusion matrix
cm = confusion_matrix(y_val, y_pred)
print('Confusion Matrix:')
print(f'  TN={cm[0][0]}  FP={cm[0][1]}')
print(f'  FN={cm[1][0]}  TP={cm[1][1]}')
print()

# classification report
print(classification_report(y_val, y_pred, target_names=['Normal (0)', 'Faulty (1)']))

=== Model Evaluation (20% Holdout) ===
Accuracy       : 0.97270
F1 Score       : 0.96591
Precision      : 0.97027
Recall         : 0.96158
ROC-AUC        : 0.99631
MCC            : 0.94317

Confusion Matrix:
  TN=5044  FP=102
  FN=133  TP=3329

              precision    recall  f1-score   support

  Normal (0)       0.97      0.98      0.98      5146
  Faulty (1)       0.97      0.96      0.97      3462

    accuracy                           0.97      8608
   macro avg       0.97      0.97      0.97      8608
weighted avg       0.97      0.97      0.97      8608



## sanity check — predict on training data itself
take TRAIN.csv, remove Class column, run through the model, compare predictions with actual labels

In [27]:
# copy train data and remove class column (treat it like unseen test data)
train_copy = pd.read_csv('TRAIN.csv')
train_copy = train_copy.drop_duplicates()  # same preprocessing as before
actual_labels = train_copy['Class'].values

# remove Class column — now it looks like test data
train_as_test = train_copy.drop(columns=['Class'])
train_test_feat_cols = [c for c in train_as_test.columns if c != 'ID']

# apply same feature engineering
X_train_test = make_features(train_as_test, train_test_feat_cols).values

# predict using the already-trained ensemble (trained on full train data)
train_preds = voting.predict(X_train_test)

# compare
match = (train_preds == actual_labels).sum()
total = len(actual_labels)
mismatch = total - match

print(f'=== Sanity Check: Predict on Training Data ===')
print(f'Total samples : {total}')
print(f'Matched       : {match}')
print(f'Mismatched    : {mismatch}')
print(f'Match rate    : {match/total*100:.4f}%')
print()

if mismatch == 0:
    print('PERFECT — all predictions match the actual labels')
else:
    print(f'{mismatch} predictions differ from actual labels')
    # show where mismatches are
    mismatched_idx = np.where(train_preds != actual_labels)[0]
    print(f'Mismatch indices (first 20): {mismatched_idx[:20]}')
    print(f'Predicted: {train_preds[mismatched_idx[:20]]}')
    print(f'Actual   : {actual_labels[mismatched_idx[:20]]}')

=== Sanity Check: Predict on Training Data ===
Total samples : 43038
Matched       : 42366
Mismatched    : 672
Match rate    : 98.4386%

672 predictions differ from actual labels
Mismatch indices (first 20): [  63  108  359  438  454  530  573  591  737  747  768  801  849 1036
 1037 1053 1075 1231 1242 1248]
Predicted: [1 0 0 0 0 1 0 0 0 0 1 1 1 1 0 0 0 1 0 0]
Actual   : [0 1 1 1 1 0 1 1 1 1 0 0 0 0 1 1 1 0 1 1]
